# Phase 3 — Daily Aggregation, Index Computation & EDA
**Project:** Jijiga Flood & Drought Risk Prediction (42.75°E, 9.25°N)  
**Weeks 9–11** | ERA5 single grid-point, 2000–2025, daily resolution

This notebook covers all Phase 3 tasks:
- **Wk 9**: Validate daily aggregated ERA5 data; recompute SPEI-6/12 (monthly scale); validate API (k=0.92) and SMI
- **Wk 10**: Full EDA — time series with EMDAT annotations, distributions, correlation, ACF/PACF, seasonal decomposition
- **Wk 11**: Construct drought and flood risk labels; create shifted target columns; class-distribution and EMDAT spot-check

**Output:** `../src/data/processed/era5_labeled.parquet` — used by Phase 4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf, adfuller
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DATA_PATH  = '../src/data/processed/era5_2000_2025.parquet'
EMDAT_PATH = '../src/data/processed/emdat.xlsx'
OUT_PATH   = '../src/data/processed/era5_labeled.parquet'

TRAIN_END = 2022   # training/test cut; reference climatology uses ≤2020
CLIM_END  = 2020   # SPEI climatology reference period end

---
## Week 9 — Data validation & index computation

In [ ]:
# ── Load & structural validation ──────────────────────────────────────────────
df = pd.read_parquet(DATA_PATH)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

expected_dates = pd.date_range('2000-01-01', '2025-12-31', freq='D')
actual_dates   = pd.DatetimeIndex(df['time'].values)
missing_dates  = expected_dates.difference(actual_dates)

print('Shape            :', df.shape)
print('Expected rows    :', len(expected_dates))
print('Date range       :', df.time.min().date(), '→', df.time.max().date())
print('Duplicate dates  :', df.time.duplicated().sum())
print('Missing dates    :', len(missing_dates))
print('Columns          :', df.columns.tolist())
df.head(3)

In [ ]:
# ── Compute SPEI-6 and SPEI-12 (monthly scale, log-logistic fit) ─────────────
# ERA5 pev is negative (upward flux), so CWB = tp + pev = tp - |pev|
df_m = df.set_index('time').resample('MS').agg(tp=('tp', 'sum'), pev=('pev', 'sum'))
df_m['cwb'] = df_m['tp'] + df_m['pev']   # climatic water balance [m/month]

def compute_monthly_spei(cwb: pd.Series, scale: int, clim_end_year: int) -> pd.Series:
    """Fit log-logistic per calendar month on training data, then standardise."""
    cwb_roll = cwb.rolling(scale, min_periods=scale).sum()
    spei_vals = np.full(len(cwb_roll), np.nan)

    for month in range(1, 13):
        m_mask    = cwb_roll.index.month == month
        train_m   = m_mask & (cwb_roll.index.year <= clim_end_year)
        x_train   = cwb_roll[train_m].dropna().values
        if len(x_train) < 5:
            continue
        shift   = x_train.min() - 1e-9
        x_tr    = x_train - shift
        try:
            c, loc, sc = stats.fisk.fit(x_tr, floc=0)
            all_m  = m_mask & cwb_roll.notna()
            x_all  = cwb_roll[all_m].values - shift
            p      = np.clip(stats.fisk.cdf(x_all, c=c, loc=loc, scale=sc), 1e-6, 1 - 1e-6)
            spei_vals[np.where(all_m)[0]] = stats.norm.ppf(p)
        except Exception:
            pass

    return pd.Series(spei_vals, index=cwb_roll.index, name=f'spei_{scale}')

df_m['spei_6']  = compute_monthly_spei(df_m['cwb'], 6,  CLIM_END)
df_m['spei_12'] = compute_monthly_spei(df_m['cwb'], 12, CLIM_END)

# Forward-fill monthly SPEI values to daily resolution
df = df.set_index('time')
df['spei_6']  = df_m['spei_6'].reindex(df.index, method='ffill')
df['spei_12'] = df_m['spei_12'].reindex(df.index, method='ffill')
df = df.reset_index()

print('SPEI-6  range:', df['spei_6'].min().round(3),  '→', df['spei_6'].max().round(3))
print('SPEI-12 range:', df['spei_12'].min().round(3), '→', df['spei_12'].max().round(3))
print('NaNs SPEI-6 :', df['spei_6'].isna().sum(), '  SPEI-12:', df['spei_12'].isna().sum())

# Plot SPEI-6 and SPEI-12 with known drought annotations
known_droughts = [2002, 2006, 2011, 2016, 2017, 2022, 2023]
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, col, label in zip(axes, ['spei_6', 'spei_12'], ['SPEI-6 (6-month)', 'SPEI-12 (12-month)']):
    s = df.set_index('time')[col].dropna()
    ax.fill_between(s.index, s.values, 0, where=s.values < 0, color='brown',  alpha=0.3, label='Deficit')
    ax.fill_between(s.index, s.values, 0, where=s.values >= 0, color='steelblue', alpha=0.3, label='Surplus')
    ax.plot(s.index, s.values, color='black', lw=0.4, alpha=0.5)
    ax.axhline(-1.0, color='orange', ls='--', lw=1, label='Moderate drought')
    ax.axhline(-1.5, color='red',    ls='--', lw=1, label='Severe drought')
    for yr in known_droughts:
        ax.axvspan(pd.Timestamp(yr, 1, 1), pd.Timestamp(yr, 12, 31),
                   alpha=0.08, color='purple', label='_nolegend_')
    ax.set_ylabel(label, fontsize=9)
    ax.legend(fontsize=7, ncol=4, loc='lower left')
axes[0].set_title('SPEI at Jijiga — purple bands = known Horn of Africa drought years', fontsize=10)
plt.tight_layout()
plt.savefig('phase3_spei.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── API with k=0.92, SMI using field-capacity normalisation ───────────────────
def compute_api(tp_series: pd.Series, decay: float = 0.92) -> pd.Series:
    vals = tp_series.to_numpy(dtype=float)
    api  = np.zeros(len(vals), dtype=float)
    for i in range(1, len(vals)):
        api[i] = decay * api[i - 1] + vals[i]
    return pd.Series(api, index=tp_series.index, name='api_92')

df['api_92'] = compute_api(df['tp'], decay=0.92)

# Field capacity for loamy sand typical of Somali region (ERA5 land)
FC1, FC2 = 0.323, 0.323   # m³/m³
df['smi_fc'] = ((df['swvl1'] + df['swvl2']) / (FC1 + FC2)).clip(0, 1)

# Verify API: after a rain event it should spike and then decay
fig, ax = plt.subplots(figsize=(14, 3))
ax2 = ax.twinx()
ax.bar(df['time'], df['tp'] * 1000, color='steelblue', alpha=0.4, width=1, label='tp [mm]')
ax2.plot(df['time'], df['api_92'], color='darkorange', lw=1, label='API (k=0.92)')
ax.set_ylabel('Daily precip [mm]', color='steelblue', fontsize=9)
ax2.set_ylabel('API [m]', color='darkorange', fontsize=9)
ax.set_title('API decay validation — spikes follow rainfall and decay exponentially', fontsize=10)
ax.set_xlim(pd.Timestamp('2010-01-01'), pd.Timestamp('2012-12-31'))
plt.tight_layout()
plt.show()

print(f'API-92  range: {df.api_92.min():.6f} → {df.api_92.max():.6f}')
print(f'SMI-FC  range: {df.smi_fc.min():.4f} → {df.smi_fc.max():.4f}')
print(f'total_ro range: {df.total_ro.min():.6f} → {df.total_ro.max():.6f}')

---
## Week 10 — Exploratory Data Analysis

In [ ]:
# ── Load EMDAT events ─────────────────────────────────────────────────────────
emdat = pd.read_excel(EMDAT_PATH)
eth   = emdat[emdat['ISO'] == 'ETH'].copy()

def to_date(row, prefix):
    try:
        yr = int(row[f'{prefix} Year'])
        mo = int(row.get(f'{prefix} Month') or 1)
        dy = int(row.get(f'{prefix} Day')   or 1)
        return pd.Timestamp(yr, mo, dy)
    except Exception:
        return pd.NaT

for dis_type in ['Flood', 'Drought']:
    sub = eth[eth['Disaster Type'] == dis_type].copy()
    sub['start_date'] = sub.apply(to_date, prefix='Start', axis=1)
    sub['end_date']   = sub.apply(to_date, prefix='End',   axis=1)
    sub = sub[sub['Start Year'].between(2000, 2025)].dropna(subset=['start_date'])
    if dis_type == 'Flood':
        eth_flood   = sub
    else:
        eth_drought = sub

print(f'ETH flood events   : {len(eth_flood)}')
print(f'ETH drought events : {len(eth_drought)}')

In [ ]:
# ── P1: Full EDA — 4-panel time series with EMDAT annotations ─────────────────
DROUGHT_BANDS = [(2002,2002),(2006,2006),(2011,2011),(2016,2017),(2022,2023)]

df_idx = df.set_index('time')
indices = [
    ('spei_6',   'SPEI-6 (6-month)',      'royalblue'),
    ('api_92',   'API (k=0.92)',           'darkorange'),
    ('smi_fc',   'SMI (field-capacity)',   'forestgreen'),
    ('total_ro', 'Total Runoff [m/day]',   'crimson'),
]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

for ax, (col, label, color) in zip(axes, indices):
    s = df_idx[col].dropna()
    ax.plot(s.index, s.values, color=color, alpha=0.25, lw=0.5)
    rm = s.rolling(30).mean()
    ax.plot(rm.index, rm.values, color=color, lw=1.5, label='30-day mean')
    ax.set_ylabel(label, fontsize=9)
    for y0, y1 in DROUGHT_BANDS:
        ax.axvspan(pd.Timestamp(y0, 1, 1), pd.Timestamp(y1, 12, 31),
                   alpha=0.10, color='saddlebrown')
    for _, row in eth_flood.iterrows():
        ax.axvline(row['start_date'], color='navy', alpha=0.25, lw=0.6)
    ax.legend(fontsize=8)

axes[0].set_title(
    'Jijiga (42.75°E, 9.25°N) — All Indices 2000–2025\n'
    'Brown bands = major Horn of Africa drought years  |  Navy lines = EMDAT Ethiopia flood events',
    fontsize=11
)
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('phase3_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── P2: Distribution analysis — histograms + KDE + ADF stationarity test ──────
from scipy.stats import gaussian_kde

plot_vars = [
    ('spei_6',   'SPEI-6',          'royalblue'),
    ('spei_12',  'SPEI-12',         'steelblue'),
    ('api_92',   'API (k=0.92)',     'darkorange'),
    ('smi_fc',   'SMI (FC-norm)',    'forestgreen'),
    ('total_ro', 'Total Runoff',     'crimson'),
    ('tp',       'Precipitation tp', 'purple'),
    ('t2m',      'Temperature t2m',  'tomato'),
    ('e',        'Evaporation e',    'teal'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (col, label, color) in zip(axes.flat, plot_vars):
    data = df[col].dropna().values.astype(float)
    ax.hist(data, bins=60, color=color, alpha=0.55, density=True)
    kde = gaussian_kde(data)
    xr  = np.linspace(data.min(), data.max(), 300)
    ax.plot(xr, kde(xr), color=color, lw=2)
    adf_p = adfuller(data, maxlag=14, autolag=None)[1]
    ax.set_title(f'{label}\nADF p={adf_p:.3f} ({"stationary" if adf_p < 0.05 else "non-stationary"})',
                 fontsize=9)
    ax.set_xlabel(col, fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('Distribution Analysis — All Indices and Key Variables', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('phase3_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── P3: Correlation matrix + ACF/PACF (up to 30 lags) ────────────────────────
corr_cols = ['spei_6', 'spei_12', 'api_92', 'smi_fc', 'total_ro', 'tp', 't2m', 'e', 'pev']
corr = df[corr_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation Matrix — Jijiga Indices & Raw Variables', fontsize=11)
plt.tight_layout()
plt.savefig('phase3_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# ACF/PACF plots
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
acf_indices = [('spei_6','SPEI-6','royalblue'),('api_92','API-92','darkorange'),
               ('smi_fc','SMI-FC','forestgreen'),('total_ro','Total Runoff','crimson')]
for i, (col, label, color) in enumerate(acf_indices):
    series = df[col].dropna().values
    nlags  = 30
    acf_v  = acf(series,  nlags=nlags, fft=True)
    pacf_v = pacf(series, nlags=nlags)
    ci = 1.96 / np.sqrt(len(series))
    lags_x = np.arange(len(acf_v))
    for ax, vals, kind in zip(axes[i], [acf_v, pacf_v], ['ACF', 'PACF']):
        ax.bar(lags_x, vals, color=color, alpha=0.7)
        ax.axhline(0,   color='black', lw=0.5)
        ax.axhline( ci, color='red',   lw=1, ls='--', label='95% CI')
        ax.axhline(-ci, color='red',   lw=1, ls='--')
        ax.set_title(f'{kind} — {label}', fontsize=10)
        ax.set_xlabel('Lag (days)')
        ax.legend(fontsize=7)

plt.suptitle('ACF and PACF up to 30 lags', fontsize=11)
plt.tight_layout()
plt.savefig('phase3_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── P4: Seasonal decomposition — period=365 (additive model) ─────────────────
decomp_indices = [
    ('spei_6',   'SPEI-6',        'royalblue'),
    ('api_92',   'API-92',        'darkorange'),
    ('smi_fc',   'SMI-FC',        'forestgreen'),
    ('total_ro', 'Total Runoff',  'crimson'),
]

fig, axes = plt.subplots(4, 4, figsize=(18, 14))

for row_i, (col, label, color) in enumerate(decomp_indices):
    series = df.set_index('time')[col].dropna().asfreq('D', method='ffill')
    result = seasonal_decompose(series, model='additive', period=365,
                                extrapolate_trend='freq')
    comps = [
        (result.observed,  'Observed'),
        (result.trend,     'Trend'),
        (result.seasonal,  'Seasonal'),
        (result.resid,     'Residual'),
    ]
    for col_j, (comp, cname) in enumerate(comps):
        ax = axes[row_i, col_j]
        ax.plot(comp.index, comp.values, color=color, lw=0.6)
        ax.set_title(f'{label} — {cname}', fontsize=8)
        ax.tick_params(labelsize=6)
        if row_i == 0 and cname == 'Trend':
            ax.set_xlabel('')

plt.suptitle('Seasonal Decomposition — period=365, additive\n'
             'Seasonal component should show bimodal Horn of Africa pattern (Apr–May, Oct–Nov peaks)',
             fontsize=10)
plt.tight_layout()
plt.savefig('phase3_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Week 11 — Risk Label Construction

In [ ]:
# ── P1: Drought risk labels — SPEI-6 thresholds + API/SMI modifier ───────────
train_mask = df['time'].dt.year <= TRAIN_END

# Compute 25th percentile cutoffs from TRAINING data only
api_p25 = df.loc[train_mask, 'api_92'].quantile(0.25)
smi_p25 = df.loc[train_mask, 'smi_fc'].quantile(0.25)
print(f'Training API 25th pct: {api_p25:.6f}')
print(f'Training SMI 25th pct: {smi_p25:.4f}')

# McKee (1993) SPEI thresholds
def spei_base_class(spei_val):
    if pd.isna(spei_val):
        return np.nan
    if spei_val >= -0.5:  return 0   # Low
    if spei_val >= -1.0:  return 1   # Moderate
    if spei_val >= -1.5:  return 2   # Elevated
    if spei_val >= -2.0:  return 3   # High
    return 4                          # Extreme

df['drought_base'] = df['spei_6'].map(spei_base_class)

# Modifier: elevate one level if base ∈ {1,2,3} AND either index below 25th pct
def apply_modifier(row):
    base = row['drought_base']
    if pd.isna(base):
        return np.nan
    base = int(base)
    if base in (1, 2, 3):
        if row['api_92'] < api_p25 or row['smi_fc'] < smi_p25:
            base = min(base + 1, 4)
    return base

df['drought_risk'] = df.apply(apply_modifier, axis=1)

# Class distribution check
dist = df['drought_risk'].value_counts(normalize=True).sort_index() * 100
label_map = {0:'Low', 1:'Moderate', 2:'Elevated', 3:'High', 4:'Extreme'}
print('\nDrought risk class distribution:')
for k, v in dist.items():
    print(f'  {int(k)} {label_map[int(k)]:10s}: {v:.1f}%')

fig, ax = plt.subplots(figsize=(6, 3))
colors5 = ['#2ecc71','#f39c12','#e67e22','#e74c3c','#8e44ad']
ax.bar([label_map[k] for k in sorted(dist.index)],
       [dist[k] for k in sorted(dist.index)], color=colors5)
ax.set_ylabel('% of days')
ax.set_title('Drought Risk Class Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# ── P2: Flood risk labels — composite score 0.40×API + 0.35×SMI + 0.25×total_ro
# Min-max normalisation parameters fitted on TRAINING data only
norm_params = {}
for col in ['api_92', 'smi_fc', 'total_ro']:
    tr_min = df.loc[train_mask, col].min()
    tr_max = df.loc[train_mask, col].max()
    norm_params[col] = (tr_min, tr_max)
    df[f'norm_{col}'] = ((df[col] - tr_min) / (tr_max - tr_min + 1e-12)).clip(0, 1)

df['flood_score'] = (0.40 * df['norm_api_92'] +
                     0.35 * df['norm_smi_fc']  +
                     0.25 * df['norm_total_ro'])

# Percentile thresholds from TRAINING data
tr_scores = df.loc[train_mask, 'flood_score']
p65, p80, p90, p97 = (tr_scores.quantile(q) for q in [0.65, 0.80, 0.90, 0.97])
print(f'Flood score percentile thresholds (training 2000–{TRAIN_END}):')
print(f'  p65={p65:.4f}  p80={p80:.4f}  p90={p90:.4f}  p97={p97:.4f}')

def flood_label(score):
    if pd.isna(score): return np.nan
    if score < p65: return 0
    if score < p80: return 1
    if score < p90: return 2
    if score < p97: return 3
    return 4

df['flood_risk'] = df['flood_score'].map(flood_label)

fdist = df['flood_risk'].value_counts(normalize=True).sort_index() * 100
print('\nFlood risk class distribution:')
for k, v in fdist.items():
    print(f'  {int(k)} {label_map[int(k)]:10s}: {v:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(axes,
    [df['drought_risk'], df['flood_risk']],
    ['Drought Risk', 'Flood Risk']):
    dist_ = data.value_counts(normalize=True).sort_index() * 100
    ax.bar([label_map[k] for k in sorted(dist_.index)],
           [dist_[k] for k in sorted(dist_.index)], color=colors5)
    ax.set_ylabel('% of days')
    ax.set_title(title)
plt.tight_layout()
plt.savefig('phase3_risk_classes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── P3: Shifted target columns for all 8 forecast tasks ──────────────────────
horizons = [1, 3, 7, 14]
for n in horizons:
    df[f'drought_risk_t_plus_{n}'] = df['drought_risk'].shift(-n)
    df[f'flood_risk_t_plus_{n}']   = df['flood_risk'].shift(-n)

# Manual spot-check (3 random dates)
np.random.seed(0)
sample_idx = np.random.randint(100, len(df) - 14, 3)
print('Shifted target spot-checks:')
for idx in sample_idx:
    date = df.loc[idx, 'time']
    target_val = df.loc[idx, 'drought_risk_t_plus_7']
    actual_val = df.loc[idx + 7, 'drought_risk']
    match = 'OK' if target_val == actual_val or (pd.isna(target_val) and pd.isna(actual_val)) else 'FAIL'
    print(f'  {date.date()} drought_risk_t+7={target_val}  drought_risk({(date + pd.Timedelta(7,"D")).date()})={actual_val}  [{match}]')

print(f'\nTarget columns created: {[c for c in df.columns if "t_plus" in c]}')
print(f'NaNs in each target (last {14} rows expected):')
for c in [c for c in df.columns if 't_plus' in c]:
    print(f'  {c}: {df[c].isna().sum()} NaN rows')

In [ ]:
# ── P4: Class distribution across all 8 targets + EMDAT spot validation ───────
target_cols = ([f'drought_risk_t_plus_{n}' for n in horizons] +
               [f'flood_risk_t_plus_{n}'   for n in horizons])

print('Class distributions for all 8 forecast targets:')
print(f'{"Task":<30} {"Low%":>6} {"Mod%":>6} {"Elev%":>6} {"High%":>6} {"Ext%":>6}')
print('-' * 65)
for col in target_cols:
    d = df[col].value_counts(normalize=True).sort_index() * 100
    vals = [d.get(k, 0) for k in range(5)]
    print(f'{col:<30} {vals[0]:>6.1f} {vals[1]:>6.1f} {vals[2]:>6.1f} {vals[3]:>6.1f} {vals[4]:>6.1f}')

# EMDAT spot validation: every major event should reach ≥ Moderate flood risk
print('\nEMDAT Flood Event Spot Check (Ethiopia):')
print(f'{"Year-Month":<12} {"MaxFloodRisk":<15} {"Location"}')
print('-' * 60)
for _, row in eth_flood.sort_values('Start Year').iterrows():
    yr = int(row['Start Year'])
    mo = int(row.get('Start Month') or 1)
    end_date = row['end_date'] if pd.notna(row['end_date']) else row['start_date'] + pd.Timedelta(30, 'D')
    window = df[(df['time'] >= row['start_date'] - pd.Timedelta(30, 'D')) &
                (df['time'] <= end_date + pd.Timedelta(30, 'D'))]
    max_risk = int(window['flood_risk'].max()) if len(window) else -1
    flag = '⚠' if max_risk < 1 else ''
    loc  = str(row.get('Location', ''))[:35]
    print(f'{yr}-{mo:02d}       {max_risk}  {label_map.get(max_risk,"?")}  {flag}  {loc}')

In [ ]:
# ── Save labeled dataset ──────────────────────────────────────────────────────
drop_tmp = ['drought_base', 'norm_api_92', 'norm_smi_fc', 'norm_total_ro', 'flood_score']
df_out = df.drop(columns=[c for c in drop_tmp if c in df.columns])

df_out.to_parquet(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df_out.shape}')
print('New columns:', [c for c in df_out.columns if c not in
       pd.read_parquet(DATA_PATH).columns.tolist()])